In [1]:
import numpy as np
import zarr

from tqdm  import tqdm

In [2]:
from zarro import (
    DataKind,
    WriteType,
    ControlVar,
    AxisSpecs,
    ReadoutSpecs,
    MeasurementSchema,
    ZarrWriter,
    NpzWriter
)

## **Examples of 2D scans**

In [3]:
Vch = ControlVar(name="Vch", values=np.linspace(0, 1, 200, dtype=np.float32))
Vm  = ControlVar(name="Vm",  values=np.logspace(np.log10(0.1), np.log10(10), 200, dtype=np.float32))
Vgt = ControlVar(name="Vgt", values=np.linspace(1, 2, 10, dtype=np.float32))

mschema = MeasurementSchema(
    control = [
        AxisSpecs([Vgt]),
        AxisSpecs([Vch, Vm])
    ],
    readout = [
        ReadoutSpecs(name="lockin_X", kind=DataKind.SCALAR, unit="V"),
        ReadoutSpecs(name="S21", kind=DataKind.TRACE, shape=(1601,), unit="dBm"),
        ReadoutSpecs(name="temperature", kind=DataKind.SCALAR, unit="K"),
    ],
    writetype=WriteType.ALL,
)

root_path = "/Users/helium/Documents/data/2026-02-24/0001"

### **2D scan / save all at once in the end of scan**

use `writer.write_all(dict)`

In [4]:
mschema.writetype = WriteType.ALL

writer = ZarrWriter(
    root=root_path,
    schema=mschema,
    overwrite=True,
    initialize_arrays=True
)

In [5]:
MAG_All = []
LOCKIN_All = []
TEMP_All = []

for i in tqdm(range(len(Vgt.values))):
    MAG = []
    LOCKIN = []
    TEMP = []
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()

        MAG.append(mag)
        LOCKIN.append(lockinx)
        TEMP.append(T1K)

    LOCKIN_All.append(LOCKIN)
    TEMP_All.append(TEMP)
    MAG_All.append(MAG)

writer.write_all({"S21": MAG_All, "temperature": TEMP_All, "lockin_X": LOCKIN_All})
writer.add_array("Freqs", freq, meta={"unit": "Hz", "description": "Frequency points for S21 measurement"})

100%|██████████| 10/10 [00:00<00:00, 300.43it/s]


In [6]:
writer.tags = ["vna", "temperature", "lockin", "chipid_1234"]
writer.write_metadata()

### **2D scan / save trace at each step**

use `writer.write_trace((i,), dict)`

In [7]:
mschema.writetype = WriteType.TRACEWISE

writer = ZarrWriter(
    root=root_path,
    schema=mschema,
    overwrite=True,
    initialize_arrays=True
)

In [8]:
for i in tqdm(range(len(Vgt.values))):
    MAG = []
    LOCKIN = []
    TEMP = []
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()

        MAG.append(mag)
        LOCKIN.append(lockinx)
        TEMP.append(T1K)

    writer.write_trace((i,), {"S21": MAG, "temperature": TEMP, "lockin_X": LOCKIN})

100%|██████████| 10/10 [00:00<00:00, 114.59it/s]


### **2D scan / save trace at each point**

use `writer.write_point((i,j), dict)`

if the writing is slow, try to adjust `writer.target_file_size_bytes` in a way, that `shard_shape` is as close to `chunk_shape`

In [9]:
mschema.writetype = WriteType.POINTWISE

writer = ZarrWriter(
    root=root_path,
    schema=mschema,
    target_file_size_bytes=1*1024*1024,  # 5 MB
    overwrite=True,
    initialize_arrays=True
)
print(writer.vault["S21"].info)

Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (10, 200, 1601)
Shard shape        : (1, 166, 1601)
Chunk shape        : (1, 1, 1601)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 12808000 (12.2M)


In [10]:
for i in tqdm(range(len(Vgt.values))):
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()
        writer.write_point((i, j), {"S21": mag, "temperature": T1K, "lockin_X": lockinx})

100%|██████████| 10/10 [00:09<00:00,  1.05it/s]


## **Examples of 1D scans**

In [11]:
Vch = ControlVar(name="Vch", values=np.linspace(0, 1, 200, dtype=np.float32))
Vm  = ControlVar(name="Vm",  values=np.logspace(np.log10(0.1), np.log10(10), 200, dtype=np.float32))

mschema = MeasurementSchema(
    control = [
        AxisSpecs([Vch, Vm])
    ],
    readout = [
        ReadoutSpecs(name="lockin_X", kind=DataKind.SCALAR, unit="V"),
        ReadoutSpecs(name="S21", kind=DataKind.TRACE, shape=(1601,), unit="dBm"),
        ReadoutSpecs(name="temperature", kind=DataKind.SCALAR, unit="K"),
    ],
    writetype=WriteType.ALL,
)

root_path = "/Users/helium/Documents/data/2026-02-24/0002"

### **1D scan / save all at once in the end of scan**

use `writer.write_all(dict)`

In [12]:
mschema.writetype = WriteType.ALL

writer = ZarrWriter(
    root=root_path,
    schema=mschema,
    overwrite=True,
    initialize_arrays=True
)

In [13]:
MAG = []
LOCKIN = []
TEMP = []
for j in range(len(Vch.values)):
    freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
    lockinx = np.random.random()
    T1K = np.random.random()

    MAG.append(mag)
    LOCKIN.append(lockinx)
    TEMP.append(T1K)

writer.write_all({"S21": MAG, "temperature": TEMP, "lockin_X": LOCKIN})

### **1D scan / save data at each point**

use `writer.write_point((i,), dict)`

if the writing is slow, try to adjust `writer.target_file_size_bytes` in a way, that `shard_shape` is as close to `chunk_shape`

In [14]:
mschema.writetype = WriteType.POINTWISE

writer = ZarrWriter(
    root=root_path,
    schema=mschema,
    target_file_size_bytes=1*1024*1024,  # 5 MB
    overwrite=True,
    initialize_arrays=True
)

In [15]:
for j in range(len(Vch.values)):
    freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
    lockinx = np.random.random()
    T1K = np.random.random()

    writer.write_point((j,), {"S21": mag, "temperature": T1K, "lockin_X": lockinx})

## **Writing npz_like**

In [16]:
Vch = ControlVar(name="Vch", values=np.linspace(0, 1, 200, dtype=np.float32))
Vm  = ControlVar(name="Vm",  values=np.logspace(np.log10(0.1), np.log10(10), 200, dtype=np.float32))
Vgt = ControlVar(name="Vgt", values=np.linspace(1, 2, 200, dtype=np.float32))

mschema = MeasurementSchema(
    control = [
        AxisSpecs([Vgt]),
        AxisSpecs([Vch, Vm])
    ],
    readout = [
        ReadoutSpecs(name="lockin_X", kind=DataKind.SCALAR, unit="V"),
        ReadoutSpecs(name="S21", kind=DataKind.TRACE, shape=(1601,), unit="dBm"),
        ReadoutSpecs(name="temperature", kind=DataKind.SCALAR, unit="K"),
    ],
    writetype=WriteType.ALL,
)

root_path = "/Users/helium/Documents/data/2026-02-24/0003"


writer = ZarrWriter(
    root=root_path,
    schema=mschema,
    initialize_arrays=False,
    overwrite=True,
)

In [17]:
writer.vault.attrs.put({"experiment": "demo", "date": "2026-02-24", "operator": "helium"})

In [18]:
MAG_All = []
LOCKIN_All = []
TEMP_All = []

for i in tqdm(range(len(Vgt.values))):
    MAG = []
    LOCKIN = []
    TEMP = []
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()

        MAG.append(mag)
        LOCKIN.append(lockinx)
        TEMP.append(T1K)

    LOCKIN_All.append(LOCKIN)
    TEMP_All.append(TEMP)
    MAG_All.append(MAG)

writer.write_npzlike({"S21": MAG_All, "temperature": TEMP_All, "lockin_X": LOCKIN_All})

writer.add_array("Freqs", freq, meta={"unit": "Hz", "description": "Frequency points for S21 measurement"})

100%|██████████| 200/200 [00:00<00:00, 450.20it/s]


In [19]:
writer.tags = ["vna", "temperature", "lockin", "chipid_1234"]
writer.write_metadata()

## **Reading data**

In [20]:
z2 = zarr.open(root_path + '/vault.zarr', mode='r')


In [21]:
list(z2.keys())

['lockin_X', 'temperature', 'S21', 'Freqs']

In [22]:
list(z2.attrs.keys())

['date', 'schema', 'tags']

In [23]:
z2["S21"][:,159, 8]

array([0.01272592, 0.27096692, 0.5738464 , 0.71203592, 0.41846388,
       0.73245591, 0.64788334, 0.54442242, 0.39611797, 0.05200457,
       0.72604269, 0.72732678, 0.44775767, 0.6483248 , 0.68518598,
       0.53124805, 0.3860271 , 0.03266137, 0.97457794, 0.05512488,
       0.25903225, 0.22721234, 0.34137255, 0.06754415, 0.88573446,
       0.2639823 , 0.87460464, 0.30824268, 0.1984823 , 0.65730943,
       0.32393378, 0.64257949, 0.17531751, 0.78890146, 0.56178155,
       0.90918578, 0.32116422, 0.3169387 , 0.99751704, 0.80114226,
       0.10975714, 0.95032368, 0.23159263, 0.99708865, 0.68500427,
       0.47822765, 0.94612111, 0.31566465, 0.15308994, 0.67703267,
       0.97457814, 0.85689225, 0.98244151, 0.35497002, 0.66815529,
       0.73177406, 0.95102631, 0.57189577, 0.07448586, 0.46377097,
       0.90605484, 0.58821211, 0.48798616, 0.01672401, 0.11726451,
       0.43732729, 0.37958086, 0.19938747, 0.21226084, 0.60517938,
       0.56799634, 0.15603911, 0.7985557 , 0.5170785 , 0.92265

## **Using NpzWriter**

In [28]:
Vch = ControlVar(name="Vch", values=np.linspace(0, 1, 200, dtype=np.float32))
Vm  = ControlVar(name="Vm",  values=np.logspace(np.log10(0.1), np.log10(10), 200, dtype=np.float32))
Vgt = ControlVar(name="Vgt", values=np.linspace(1, 2, 200, dtype=np.float32))

mschema = MeasurementSchema(
    control = [
        AxisSpecs([Vgt]),
        AxisSpecs([Vch, Vm])
    ],
    readout = [
        ReadoutSpecs(name="lockin_X", kind=DataKind.SCALAR, unit="V"),
        ReadoutSpecs(name="S21", kind=DataKind.TRACE, shape=(1601,), unit="dBm"),
        ReadoutSpecs(name="temperature", kind=DataKind.SCALAR, unit="K"),
    ],
    writetype=WriteType.ALL,
)

root_path = "/Users/helium/Documents/data/2026-02-24/0004"

writer = NpzWriter(
    root=root_path,
    schema=mschema,
    compressed=False
)

In [29]:
MAG_All = []
LOCKIN_All = []
TEMP_All = []

for i in tqdm(range(len(Vgt.values))):
    MAG = []
    LOCKIN = []
    TEMP = []
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()

        MAG.append(mag)
        LOCKIN.append(lockinx)
        TEMP.append(T1K)

    LOCKIN_All.append(LOCKIN)
    TEMP_All.append(TEMP)
    MAG_All.append(MAG)

writer.write({"S21": MAG_All, "temperature": TEMP_All, "lockin_X": LOCKIN_All})

#writer.add_array("Freqs", freq, meta={"unit": "Hz", "description": "Frequency points for S21 measurement"})

100%|██████████| 200/200 [00:00<00:00, 476.76it/s]


In [30]:
flist = ControlVar(name="freqs", values=freq)

mschema = MeasurementSchema(
    control = [
    ],
    readout = [
        ReadoutSpecs(name="S34", kind=DataKind.IMAGE, shape=(3, 1601,), unit=["Hz", "dBm", "deg"], contains=["f", "mag", "phase"]),
    ],
    writetype=WriteType.ALL,
)

root_path2 = "/Users/helium/Documents/data/2026-02-24/0005"

writer2 = ZarrWriter(
    root=root_path2,
    schema=mschema,
    overwrite=True,
)

writer2.write_npzlike(
    data = {"S34": np.asarray([freq, mag, phase])}
)
writer2.write_metadata()

In [31]:
z3 = np.load(root_path + "/data.npz")

In [32]:
z3

NpzFile '/Users/helium/Documents/data/2026-02-24/0004/data.npz' with keys: S21, temperature, lockin_X

In [33]:
z3["S21"][:, 145, 5]

array([0.21204016, 0.15805334, 0.26329248, 0.51343759, 0.14820108,
       0.52979302, 0.30969197, 0.2229643 , 0.64172682, 0.68019559,
       0.02581201, 0.6019382 , 0.98193588, 0.67501643, 0.25121519,
       0.97448416, 0.99808047, 0.14612637, 0.10215667, 0.20847771,
       0.78671681, 0.81033973, 0.68405661, 0.44740168, 0.07647503,
       0.67336382, 0.55651136, 0.4943668 , 0.52783856, 0.00810324,
       0.02380725, 0.78842822, 0.34000911, 0.67747696, 0.13392115,
       0.92055498, 0.4742783 , 0.64584689, 0.62509598, 0.18899801,
       0.29362396, 0.12911384, 0.71106571, 0.85474749, 0.71675544,
       0.84094941, 0.08188709, 0.07535336, 0.88768662, 0.56219512,
       0.4429111 , 0.32080758, 0.12105523, 0.60953052, 0.30755201,
       0.61630516, 0.70827434, 0.40280439, 0.32186777, 0.70414641,
       0.06988118, 0.77093563, 0.94807667, 0.88346183, 0.74900646,
       0.17370809, 0.49211146, 0.98424155, 0.59190042, 0.52631792,
       0.12615133, 0.64797942, 0.02795121, 0.60864236, 0.87030

In [34]:
z4 = zarr.open("/Users/helium/Documents/data/2026-02-24/0005" + "/vault.zarr", mode='r')

In [35]:
z4.tree()

/
└── S34 (3, 1601) float64

In [36]:
z4["S34"][:,159]

array([0.66661996, 0.68326692, 0.36258246])